# Taller de código 2: optimización local

Resolverás el problema de las **8 reinas** con Hill Climbing y Simulated Annealing.

Un estado es una lista donde el índice representa la columna y el valor representa la fila de cada reina. El costo es el número de pares de reinas que se atacan. Buscamos minimizarlo hasta llegar a cero.

Tiempo estimado: **60–75 minutos**.

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
N = 8

In [ ]:
def plot_board(state, title=None):
    n = len(state)
    board = np.fromfunction(lambda row, col: (row + col) % 2, (n, n))
    plt.figure(figsize=(5, 5))
    plt.imshow(board, cmap="Greys", vmin=0, vmax=1)
    for col, row in enumerate(state):
        plt.text(col, row, "♛", ha="center", va="center", fontsize=28, color="#c0392b")
    plt.xticks(range(n)); plt.yticks(range(n))
    plt.title(title or str(state))
    plt.show()

initial_state = [2, 1, 4, 1, 7, 7, 7, 6]
plot_board(initial_state, "Estado inicial")

## Actividad 1 — Función de costo

Cuenta cuántos pares de reinas comparten fila o diagonal. Cada pareja debe contarse una sola vez.

In [ ]:
def conflicts(state):
    """Número de pares de reinas que se atacan."""
    n = len(state)
    count = 0
    for col1 in range(n):
        for col2 in range(col1 + 1, n):
            row1, row2 = state[col1], state[col2]
            if row1 == row2 or abs(row1 - row2) == abs(col1 - col2):
                count += 1
    return count

In [ ]:
assert conflicts([0, 1, 2, 3]) == 6
assert conflicts([1, 3, 0, 2]) == 0
assert conflicts([0, 0, 2, 3]) > 0
print("Pruebas de costo superadas.")

## Actividad 2 — Generación de vecinos

Un vecino se obtiene moviendo una reina a otra fila dentro de su misma columna.

In [ ]:
def get_neighbors(state):
    """Genera todos los estados alcanzables moviendo exactamente una reina."""
    n = len(state)
    neighbors = []
    for col in range(n):
        current_row = state[col]
        for row in range(n):
            if row != current_row:
                new_state = list(state)
                new_state[col] = row
                neighbors.append(new_state)
    return neighbors

In [ ]:
neighbors = get_neighbors([0, 1, 2, 3])
assert len(neighbors) == 4 * 3
assert all(sum(a != b for a, b in zip([0, 1, 2, 3], n)) == 1 for n in neighbors)
print("Pruebas de vecinos superadas.")

## Actividad 3 — Hill Climbing

In [ ]:
def best_neighbor(state):
    """Devuelve un vecino de costo mínimo y su costo."""
    neighbors = get_neighbors(state)
    scored = [(conflicts(n), n) for n in neighbors]
    min_cost = min(c for c, _ in scored)
    best_candidates = [n for c, n in scored if c == min_cost]
    return random.choice(best_candidates), min_cost


def hill_climbing(initial_state, max_steps=200):
    """Minimiza conflicts hasta solución o mínimo local."""
    current_state = list(initial_state)
    current_cost = conflicts(current_state)
    history_costs = [current_cost]

    for _ in range(max_steps):
        if current_cost == 0:
            break
        next_state, next_cost = best_neighbor(current_state)
        if next_cost < current_cost:
            current_state = next_state
            current_cost = next_cost
            history_costs.append(current_cost)
        else:
            # Mínimo local alcanzado (no hay mejora estricta)
            break

    return current_state, history_costs

In [ ]:
random.seed(RANDOM_STATE)
state_hc, costs_hc = hill_climbing(initial_state)
print("Estado final:", state_hc)
print("Costo final:", conflicts(state_hc))
print("Pasos:", len(costs_hc) - 1)

plt.plot(costs_hc, marker="o")
plt.xlabel("Iteración"); plt.ylabel("Conflictos")
plt.title("Hill Climbing")
plt.show()

## Actividad 4 — Simulated Annealing

Estamos minimizando. Define:

$$\Delta = C_{nuevo}-C_{actual}.$$

Si $\Delta\leq 0$, el movimiento no empeora. Si $\Delta>0$, se acepta con probabilidad:

$$P=\exp\left(-\frac{\Delta}{T}\right).$$

In [ ]:
def acceptance_probability(current_cost, new_cost, temperature):
    delta = new_cost - current_cost
    if delta <= 0:
        return 1.0
    return math.exp(-delta / temperature)


def simulated_annealing(initial_state, initial_temperature=10.0,
                        cooling_rate=0.995, min_temperature=1e-3,
                        max_steps=10_000):
    current_state = list(initial_state)
    current_cost = conflicts(current_state)
    history_costs = [current_cost]
    temperature = initial_temperature

    for _ in range(max_steps):
        if current_cost == 0 or temperature < min_temperature:
            break
        neighbors = get_neighbors(current_state)
        candidate = random.choice(neighbors)
        candidate_cost = conflicts(candidate)

        prob = acceptance_probability(current_cost, candidate_cost, temperature)
        if random.random() < prob:
            current_state = candidate
            current_cost = candidate_cost

        history_costs.append(current_cost)
        temperature *= cooling_rate

    return current_state, history_costs

In [ ]:
assert acceptance_probability(5, 4, 1.0) == 1.0
assert 0 < acceptance_probability(4, 5, 1.0) < 1

random.seed(RANDOM_STATE)
state_sa, costs_sa = simulated_annealing(initial_state)
print("Estado final:", state_sa)
print("Costo final:", conflicts(state_sa))
plot_board(state_sa, f"Simulated Annealing — costo {conflicts(state_sa)}")

## Actividad 5 — Comparación en varios reinicios

In [ ]:
# Comparación de algoritmos en 30 reinicios independientes
n_runs = 30
hc_success, sa_success = 0, 0
hc_final_costs, sa_final_costs = [], []

for seed in range(n_runs):
    random.seed(seed + 100)
    start_state = [random.randint(0, 7) for _ in range(8)]
    
    # Hill Climbing
    end_hc, costs_h = hill_climbing(start_state)
    c_hc = conflicts(end_hc)
    hc_final_costs.append(c_hc)
    if c_hc == 0:
        hc_success += 1
        
    # Simulated Annealing
    end_sa, costs_s = simulated_annealing(start_state)
    c_sa = conflicts(end_sa)
    sa_final_costs.append(c_sa)
    if c_sa == 0:
        sa_success += 1

results_opt = pd.DataFrame([
    {
        "Algoritmo": "Hill Climbing (Steepest Ascent)",
        "Tasa Éxito (costo 0)": f"{hc_success / n_runs * 100:.1f}% ({hc_success}/{n_runs})",
        "Costo Final Promedio": f"{sum(hc_final_costs) / n_runs:.2f}",
        "Mínimo Costo": min(hc_final_costs)
    },
    {
        "Algoritmo": "Simulated Annealing",
        "Tasa Éxito (costo 0)": f"{sa_success / n_runs * 100:.1f}% ({sa_success}/{n_runs})",
        "Costo Final Promedio": f"{sum(sa_final_costs) / n_runs:.2f}",
        "Mínimo Costo": min(sa_final_costs)
    }
])
results_opt

## Preguntas de análisis y respuestas

1. **¿Por qué Hill Climbing puede detenerse con costo mayor que cero?**
   - Porque se queda atrapado en un **mínimo local** o meseta (plateau) donde ninguno de sus vecinos inmediatos mejora estrictamente el costo actual, deteniendo su ejecución prematuramente aunque no sea el óptimo global.

2. **¿Qué permite a Simulated Annealing escapar de mínimos locales?**
   - El mecanismo probabilístico de aceptación basado en la distribución de Boltzmann: permite aceptar movimientos cuesta arriba ($\Delta > 0$, que empeoran la función objetivo) con probabilidad $P = \exp(-\Delta / T)$, lo que le otorga la energía estocástica necesaria para salir de pozos locales.

3. **Si el enfriamiento es demasiado rápido, ¿a cuál algoritmo empieza a parecerse?**
   - Empieza a comportarse prácticamente como **Hill Climbing voraz** (Greedy Local Search), pues la temperatura decae a valores cercanos a cero casi de inmediato, anulando la probabilidad de aceptar empeoramientos ($P 	o 0$).

4. **¿Qué ocurre si la temperatura permanece muy alta?**
   - El algoritmo se convierte en un **paseo aleatorio puro** (Random Walk), ya que $P = \exp(-\Delta / T) pprox \exp(0) = 1$, aceptando casi cualquier movimiento independientemente de si mejora o empeora drásticamente la solución.

## Checklist

- [x] Implementé costo y vecinos.
- [x] Hill Climbing acepta solo mejoras.
- [x] Simulated Annealing puede aceptar empeoramientos.
- [x] Comparé ambos algoritmos en múltiples ejecuciones.